# Data ETL and exploration/preprocessing

## Loading and Exploring Data

### Guidelines

1. Load all the kcmetro text data independently and drop irrelevant information
2. Merge all data 
3. Explore data and calendar/caldates/service id in particular
4. Create function to calculate frequencies from schedule data
5. create function to map routes that match frequency target 

Libraries to Use:

-numpy and pandas 
-geopandas and folium
-datetime 
-os and shutils 
-arcpy

In [36]:
###imports
%pip install folium pandas numpy geopandas

import numpy as np, pandas as pd
import folium as fl, geopandas as gp 
from datetime import datetime


Note: you may need to restart the kernel to use updated packages.


In [37]:
###loading data
trips = pd.read_csv(r'../data/KCM_data/trips.txt')
stop_times = pd.read_csv(r'../data/KCM_data/stop_times.txt')
stops = pd.read_csv(r'../data/KCM_data/stops.txt')
routes = pd.read_csv(r'../data/KCM_data/routes.txt')
cal = pd.read_csv(r'../data/KCM_data/calendar.txt')
cal_dates = pd.read_csv(r'../data/KCM_data/calendar_dates.txt')

In [ ]:
###dropping unnecessary information

trips_clean = trips[['route_id','service_id','trip_id','trip_headsign', 'direction_id','block_id','shape_id']]
stop_times_clean = stop_times[['trip_id','arrival_time','departure_time','stop_id','stop_sequence','shape_dist_traveled','timepoint']]
stops_clean = stops[['stop_id','stop_code','stop_name','stop_lat','stop_lon']]
routes_clean = routes[['route_id','agency_id','route_short_name','route_desc','route_type','route_color','route_text_color']]

###cal and calendar dates are to remain unchanged until I explore the data more thoroughly 
###set the variables to the clean dataframes

trips = trips_clean
stop_times = stop_times_clean
stops = stops_clean
routes = routes_clean


###find and set types
trips.service_id = trips.service_id.astype(int)
stop_times.arrival_time = pd.to_timedelta(stop_times.arrival_time)
stop_times.departure_time = pd.to_timedelta(stop_times.departure_time)



In [109]:
###merge together data

#merge all files one by one 

data_merged = pd.merge(trips, stop_times, how='left', on='trip_id')
data_merged = pd.merge(data_merged, stops, how='left', on='stop_id')
data_merged = pd.merge(data_merged, routes, how='left', on='route_id')

###filter to only kcm bus routes 
data_merged = data_merged.loc[data_merged.agency_id == 1].drop('agency_id', axis=1)
data_merged

,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,arrival_time,departure_time,stop_id,...,timepoint,stop_code,stop_name,stop_lat,stop_lon,route_short_name,route_desc,route_type,route_color,route_text_color
0,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:02:00,0 days 05:02:00,1530,...,1,1530,S Jackson St & 5th Ave S,47.599266,-122.328384,1,Kinnear - Downtown Seattle,3,FDB71A,000000
1,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:03:43,0 days 05:03:43,1610,...,0,1610,Prefontaine Pl S & Yesler Way,47.601658,-122.329941,1,Kinnear - Downtown Seattle,3,FDB71A,000000
2,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:05:24,0 days 05:05:24,538,...,0,538,3rd Ave & Columbia St,47.604095,-122.332504,1,Kinnear - Downtown Seattle,3,FDB71A,000000
3,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:07:19,0 days 05:07:19,558,...,0,558,3rd Ave & Seneca St,47.607052,-122.335213,1,Kinnear - Downtown Seattle,3,FDB71A,000000
4,100001,151056,562518401,Kinnear Seattle Center W,0,8222244,11001004,0 days 05:09:00,0 days 05:09:00,575,...,1,575,3rd Ave & Pike St,47.609642,-122.337585,1,Kinnear - Downtown Seattle,3,FDB71A,000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1107804,7994,63424,916940,NaN,1,36926,63424,0 days 07:12:00,0 days 07:12:00,200,...,1,200,1st Ave & Pine St,47.609692,-122.340632,Easy Loop,First Hill-Pioneer Square-Downtown Seattle-Bel...,3,NaN,NaN
1107805,7994,63424,916940,NaN,1,36926,63424,0 days 07:15:00,0 days 07:15:00,230,...,1,230,1st Ave & Bell St,47.613752,-122.346884,Easy Loop,First Hill-Pioneer Square-Downtown Seattle-Bel...,3,NaN,NaN
1107806,7994,63424,916940,NaN,1,36926,63424,0 days 07:22:00,0 days 07:22:00,900,...,1,900,Virginia St & 9th Ave,47.616547,-122.335388,Easy Loop,First Hill-Pioneer Square-Downtown Seattle-Bel...,3,NaN,NaN
1107807,7994,63424,916940,NaN,1,36926,63424,0 days 07:27:00,0 days 07:27:00,11075,...,1,11075,Boren Ave & Seneca St,47.610374,-122.326210,Easy Loop,First Hill-Pioneer Square-Downtown Seattle-Bel...,3,NaN,NaN


### Data Exploration

In [110]:
data_merged.loc[data_merged.service_id == 30098]

,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id,arrival_time,departure_time,stop_id,...,timepoint,stop_code,stop_name,stop_lat,stop_lon,route_short_name,route_desc,route_type,route_color,route_text_color
3295,100001,30098,651985241,Downtown Seattle,1,8221929,20001005,1 days 00:23:00,1 days 00:23:00,2010,...,1,2010,10th Ave W & W Armour St,47.645111,-122.370277,1,Kinnear - Downtown Seattle,3,FDB71A,000000
3296,100001,30098,651985241,Downtown Seattle,1,8221929,20001005,1 days 00:23:25,1 days 00:23:25,2020,...,0,2020,10th Ave W & W Bothwell St,47.643486,-122.370308,1,Kinnear - Downtown Seattle,3,FDB71A,000000
3297,100001,30098,651985241,Downtown Seattle,1,8221929,20001005,1 days 00:23:50,1 days 00:23:50,2030,...,0,2030,10th Ave W & W Halladay St,47.641792,-122.370171,1,Kinnear - Downtown Seattle,3,FDB71A,000000
3298,100001,30098,651985241,Downtown Seattle,1,8221929,20001005,1 days 00:24:27,1 days 00:24:27,2050,...,0,2050,10th Ave W & W Mcgraw St,47.639366,-122.370415,1,Kinnear - Downtown Seattle,3,FDB71A,000000
3299,100001,30098,651985241,Downtown Seattle,1,8221929,20001005,1 days 00:24:54,1 days 00:24:54,2060,...,0,2060,10th Ave W & W Crockett St,47.637585,-122.370438,1,Kinnear - Downtown Seattle,3,FDB71A,000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1107148,102769,30098,851783381,Mercer Island Station Issaquah Highlands Park ...,1,8224406,30215002,0 days 19:27:17,0 days 19:27:17,65150,...,0,65150,Highlands Dr NE & NE Ellis Dr,47.541138,-122.018890,215,Mercer Island - North Bend,3,FDB71A,000000
1107149,102769,30098,851783381,Mercer Island Station Issaquah Highlands Park ...,1,8224406,30215002,0 days 19:28:00,0 days 19:28:00,64566,...,1,64566,Issaquah Highlands Park & Ride - Bay 3,47.545704,-122.019318,215,Mercer Island - North Bend,3,FDB71A,000000
1107150,102769,30098,851783381,Mercer Island Station Issaquah Highlands Park ...,1,8224406,30215002,0 days 19:28:52,0 days 19:28:52,64549,...,0,64549,9th Ave NE & NE Ellis Dr,47.540581,-122.020340,215,Mercer Island - North Bend,3,FDB71A,000000
1107151,102769,30098,851783381,Mercer Island Station Issaquah Highlands Park ...,1,8224406,30215002,0 days 19:42:00,0 days 19:42:00,67019,...,1,67019,I-90 Expressway Ramp & 142nd Pl SE - Bay 3,47.579609,-122.151711,215,Mercer Island - North Bend,3,FDB71A,000000


In [104]:
###find data by days to calculate weekend, holiday, weekday frequencies/etc 

###find how many trips per service ID to decide which is the main one per each route
cal.loc[cal.service_id.astype(int) == 30098]
data_merged.groupby('route_short_name').service_id.unique()
###service ids seem to work generally as follows for buses: 151056 = weekday, 34059 == sun, 30098 == sat


route_short_name
1                                 [151056, 30098, 34059]
10                                [151056, 34059, 30098]
101                               [151056, 30098, 34059]
102                                             [151056]
105                               [151056, 30098, 34059]
                                           ...          
South Lake Union Streetcar        [149182, 30098, 33797]
Trailhead Direct Issaquah Alps               [242, 1884]
Trailhead Direct Mt. Si                     [242, 34059]
Valley Shuttle                    [151056, 30098, 34059]
Waterfront Shuttle                    [6236, 1877, 4015]
Name: service_id, Length: 145, dtype: object

### Frequency Calculations/Functions and Verification

In [73]:
##lets use groupby in pandas to filter/sort the data we desire and limit it to a single stop + dir + service_id 
data_merged = data_merged.sort_values(by=['stop_sequence', 'arrival_time'])

data_merged['prev_depart'] = data_merged.groupby(['route_id', 'service_id', 'direction_id', 'stop_sequence']).departure_time.shift(1)
data_merged['gap'] = data_merged['departure_time'] - data_merged['prev_depart']
route_freq = data_merged.groupby('route_short_name').gap.mean().sort_values(ascending=True)

route_freq[:10]

route_short_name
G Line                       0 days 00:10:08.729701
E Line                       0 days 00:12:15.611985
36                           0 days 00:12:40.525348
A Line                       0 days 00:13:41.744216
South Lake Union Streetcar   0 days 00:13:58.621899
7                            0 days 00:14:00.556345
72                           0 days 00:14:35.717729
Waterfront Shuttle                  0 days 00:15:00
B Line                       0 days 00:15:01.652895
H Line                       0 days 00:15:05.484830
Name: gap, dtype: timedelta64[us]

In [80]:
###function for gapping !!! FINISH !!! and test

def freq_calc(df, label='departure_time', start_time=pd.Timedelta(hours=0), end_time = pd.Timedelta(hours=28)):
    range = pd.timedelta_range(start_time, end_time)
    df_lite = df[['route_id','trip_id','stop_id','route_short_name', label]]
    df_lite = df_lite.loc[df_lite.departure_time.isin(range)]
    

### Displaying Data for User